In [1]:
import pandas as pd

df = pd.read_csv('bridges.csv', na_values=['?', ''])
df.head()

,Id,river,location,erected,purpose,length,lanes,clear-g,t-or-d,material,span,rel-l,type
0,E1,M,3.0,1818,HIGHWAY,NaN,2.0,N,THROUGH,WOOD,SHORT,S,WOOD
1,E2,A,25.0,1819,HIGHWAY,1037.0,2.0,N,THROUGH,WOOD,SHORT,S,WOOD
2,E3,A,39.0,1829,AQUEDUCT,NaN,1.0,N,THROUGH,WOOD,NaN,S,WOOD
3,E5,A,29.0,1837,HIGHWAY,1000.0,2.0,N,THROUGH,WOOD,SHORT,S,WOOD
4,E6,M,23.0,1838,HIGHWAY,NaN,2.0,N,THROUGH,WOOD,NaN,S,WOOD


In [2]:
print('missing:', df['length'].isna().sum(), 'of', len(df))
df['length'].describe()

missing: 27 of 108


count      81.000000
mean     1567.469136
std       747.491523
min       804.000000
25%      1000.000000
50%      1300.000000
75%      2000.000000
max      4558.000000
Name: length, dtype: float64

# length assumption
Bridges can be very short or very long. I am going to assume that a bridge must be at least 20 ft to be a bridge, but not longet than 10000 ft to rule out anything unreasonable for the time.

In [3]:
LOW, HIGH = 20, 10000
range_flags = df[(df['length'] < LOW) | (df['length'] > HIGH)]
range_flags[['Id', 'river', 'erected', 'purpose', 'length', 'material', 'span', 'type']]

,Id,river,erected,purpose,length,material,span,type


# Nothing got flagged.
A fixed range didn't catch anything. I'll assume a bridge length is implausible if it falls outside 1.5×IQR from the middle 50% of values (the standard statistical outlier definition).

In [4]:
q1, q3 = df['length'].quantile([0.25, 0.75])
iqr = q3 - q1
lo_fence, hi_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f'Q1={q1}, Q3={q3}, IQR={iqr}')
print(f'fences: [{lo_fence}, {hi_fence}]')

iqr_flags = df[(df['length'] < lo_fence) | (df['length'] > hi_fence)]
iqr_flags[['Id', 'river', 'erected', 'purpose', 'length', 'material', 'span', 'type']]

Q1=1000.0, Q3=2000.0, IQR=1000.0
fences: [-500.0, 3500.0]


,Id,river,erected,purpose,length,material,span,type
31,E34,O,1888,RR,4558.0,STEEL,LONG,SIMPLE-T
44,E46,A,1897,RR,4000.0,STEEL,LONG,SIMPLE-T
104,E91,O,1975,HIGHWAY,3756.0,STEEL,LONG,ARCH


Anything more than 1.5×IQR below Q1 or above Q3 got flagged as a statistical outlier. There are only 3 outliers. I will leave them becuase, while they are longer, they are all plausible lengths. E91 is only 256 feet from the cutoff. It seems fine. E34 and E46 are on the longer side for sure given when they were made, but seem very possible.